# Simulate a micrograph from PDB 6DRV

End-to-end walkthrough of the TeamTomo simulation stack:

1. Download apoferritin (**6DRV**) with `mmdf`
2. Annotate bonding, rotate, and center with `torch-structure-manipulation`
3. Build a 3D electrostatic potential with `torch-calculate-electrostatic-potential`
4. Propagate with multislice in `torch-scattering`
5. Form a CTF-modulated, dose-scaled micrograph with `torch-simulate-image`

Apoferritin is large (~120 Å). This notebook uses a **2 Å** voxel size so the 3D grid stays manageable on CPU.

In [ ]:
import mmdf
import torch
from matplotlib import pyplot as plt
from torch_calculate_electrostatic_potential import (
    GridConfig,
    default_sublattice_radius,
    potential_from_structure_3d,
)
from torch_scattering import multislice
from torch_structure_manipulation import (
    AtomicStructure,
    annotate_bonding_environments,
    apply_rotation,
    center_structure,
    create_rotation_matrix_from_euler,
)

from torch_simulate_image import (
    CtfConfig,
    FluenceConfig,
    MicrographSimulationConfig,
    PoissonConfig,
    simulate_micrograph,
)

## Simulation parameters

In [ ]:
PIXEL_SIZE = 2.0  # Å — coarse for a fast demo of betagalactosidase
VOLTAGE_KV = 300.0
PADDING_A = 10.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 1. Download PDB 6DRV

`mmdf.read("pdb:…")` fetches the entry (cached locally) and returns an atomic DataFrame with `x, y, z` columns in Angstroms.

In [ ]:
atoms = mmdf.read("pdb:6drv")
print(f"{len(atoms)} atoms, columns={list(atoms.columns)}")
atoms.head()

## 2. Bonded environments, rotation, and centering

Annotate local bonding for Peng bonded scattering factors, center at the origin, then apply a small Euler rotation about that center so the pose is not about an arbitrary PDB origin.

In [ ]:
annotated = annotate_bonding_environments(atoms, include_hydrogens=False)

# Center first so the subsequent rotation is about the molecule origin.
centered = center_structure(annotated, center_point=(0.0, 0.0, 0.0), zyx=False)
rotation = create_rotation_matrix_from_euler(
    torch.tensor([0.0, 30.0, 0.0]),
    order="ZYZ",
    degrees=True,
)
posed = apply_rotation(centered, rotation, zyx=False)

structure = AtomicStructure.from_dataframe(posed, device=DEVICE)
print(f"{structure.num_atoms} atoms on {structure.device}")
print(
    "position extent (z,y,x) Å:",
    structure.positions_zyx.amin(0),
    "→",
    structure.positions_zyx.amax(0),
)

## 3. Electrostatic potential (ESP)

Build a cubic grid from the structure bounding box (plus padding) and sample the 3D potential in **volts** with bonded Peng factors.

In [ ]:
positions = structure.positions_zyx
mins = positions.amin(dim=0) - PADDING_A
maxs = positions.amax(dim=0) + PADDING_A

grid = GridConfig.from_voxel_size_and_corner_points(
    voxel_size=(PIXEL_SIZE, PIXEL_SIZE, PIXEL_SIZE),
    left_bottom_point=tuple(mins.tolist()),
    right_upper_point=tuple(maxs.tolist()),
    sublattice_radius=default_sublattice_radius(PIXEL_SIZE),
    device=DEVICE,
)
print("grid shape (Z,Y,X):", tuple(int(n) for n in grid.grid_shape.tolist()))

potential = potential_from_structure_3d(
    structure,
    grid,
    scattering_factors="peng_bonded",
    bonded_fallback="elemental",
)
potential.shape, potential.dtype

In [ ]:
mid_z = potential.shape[0] // 2
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(potential[mid_z].detach().cpu().numpy(), cmap="magma")
ax.set_title(f"ESP central Z slice (V), z={mid_z}")
ax.set_xlabel("X")
ax.set_ylabel("Y")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 4. Exit wave (multislice)

`torch_scattering.multislice` propagates a plane wave through the potential volume and returns a complex exit wave `ψ` of shape `(H, W)`.

In [ ]:
exit_wave = multislice(potential, pixel_size=PIXEL_SIZE, voltage=VOLTAGE_KV)
exit_wave.shape, exit_wave.dtype

In [ ]:
amp = exit_wave.abs().detach().cpu().numpy()
phase = exit_wave.angle().detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(amp, cmap="gray")
axes[0].set_title("|ψ|")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(phase, cmap="twilight")
axes[1].set_title("arg(ψ)")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 5. Simulate the micrograph

`simulate_micrograph` applies the optics / detector pipeline:

```text
ψ → [aperture] → CTF → |ψ|² → envelopes / dose weight → fluence → Poisson → DQE
```

DQE is left off here (no MTF curve supplied).

In [ ]:
config = MicrographSimulationConfig(
    pixel_size=PIXEL_SIZE,
    ctf=CtfConfig(defocus_um=1.5, voltage_kv=VOLTAGE_KV),
    fluence=FluenceConfig(dose_e_per_A2=30.0),
    poisson=PoissonConfig(apply=True, deterministic=True, seed=0),
)

micrograph = simulate_micrograph(exit_wave, config)
micrograph_clean = simulate_micrograph(
    exit_wave,
    config.model_copy(update={"poisson": PoissonConfig(apply=False)}),
)
micrograph.shape, float(micrograph.mean()), float(micrograph.std())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(micrograph_clean.detach().cpu().numpy(), cmap="gray")
axes[0].set_title("Expected counts (no Poisson)")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(micrograph.detach().cpu().numpy(), cmap="gray")
axes[1].set_title("Poisson-sampled micrograph")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()